# 03 · Modelling: classical ML vs deep learning (M3 + M5 + M6)
Run `python run_pipeline.py` once (about 30–60 min on a laptop; the notebook reads its saved results).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))          # lets the notebook import the project's src/ package
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

## 1. Features & the leakage guard
Every feature uses only windows ≤ *t*; targets are shifted forward per road. The split is **chronological** (Jan–Mar train, Apr validation, May test).

In [ ]:
from src.features import build_supervised, split_by_time, FEATURES, GROUPS
df = pd.read_parquet(ROOT/"data/processed/flowcast_clean.parquet")
sup = build_supervised(df, h=1)
tr, va, te = split_by_time(sup)
print(len(FEATURES), "features;", len(tr), len(va), len(te), "rows (train/val/test)")
{g: len(c) for g, c in GROUPS.items()}

## 2. Linear regression from scratch (NumPy gradient descent)
Cost J(w,b) = mean((Xw + b − y)²); gradients ∂J/∂w = (2/n)·Xᵀ(Xw + b − y). The curve below is the cost falling during training.

In [ ]:
display(Image(str(ROOT/"reports/figures/gd_loss_curve.png")))

## 3. Scoreboard (test window)

In [ ]:
sc = pd.read_csv(ROOT/"reports/scoreboard_classical.csv")
dl = pd.read_csv(ROOT/"reports/scoreboard_dl.csv")
allsc = pd.concat([sc, dl])
for task in ["volume","travel_time","congestion","accident"]:
    print("\n==", task)
    t = allsc[(allsc.task==task)&(allsc.split=="te")].dropna(axis=1, how="all")
    display(t.drop(columns=["task","split","best_params"], errors="ignore").set_index("model").round(4))

## 4. Diagnostics

In [ ]:
for f in ["confusion_congestion","roc_accident","fi_volume","fi_accident","volume_error_breakdown","volume_residuals","pred_vs_actual_NL-007","dl_loss_curves"]:
    display(Image(str(ROOT/"reports"/"figures"/f"{f}.png")))

## 5. Read the results
See `reports/FlowCast_Final_Report.md` for the written model-selection justification, the classical-vs-deep verdict and the honest discussion of the accident-risk ceiling.